In [22]:
import pandas as pd
from nrclex import NRCLex
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [23]:
df_songs = pd.read_csv('dataset.csv')

In [24]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer

# 1. Initialize the lemmatizer
# You might need to run: nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

def lemmatized_tokenizer(text):
    # Tokenize using scikit-learn's default pattern
    tokens = nltk.word_tokenize(text)
    # Lemmatize each word
    return [lemmatizer.lemmatize(w) for w in tokens]

# 1. Vectorize text (using raw counts for LDA)
count_vectorizer = CountVectorizer(tokenizer=lemmatized_tokenizer, stop_words='english', max_features=9000)
X_counts = count_vectorizer.fit_transform(df_songs['lyrics_cleaned'])

# 2. Initialize LDA
# n_components is the number of topics you want to find
lda = LatentDirichletAllocation(n_components=20, random_state=42)
lda_topics = lda.fit_transform(X_counts)

# 'lda_topics' now contains the topic weight for every song
df_songs['top_topic'] = lda_topics.argmax(axis=1)

C:\Users\conno\AppData\Roaming\Python\Python313\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
C:\Users\conno\AppData\Roaming\Python\Python313\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ha', 'u', 'wa'] not in stop_words.
  warnings.warn(


In [25]:
# 2. Create column names like 'topic_0', 'topic_1', etc.
topic_columns = [f'topic_{i}' for i in range(lda.n_components)]

# 3. Create a temporary DataFrame for the weights
df_weights = pd.DataFrame(lda_topics, columns=topic_columns, index=df_songs.index)

In [26]:
# 4. Join them back to your original DataFrame
df_songs = pd.concat([df_songs, df_weights], axis=1)

In [27]:
def print_top_words(model, feature_names, n_top_words):
    for topic_idx, topic in enumerate(model.components_):
        message = f"Topic #{topic_idx}: "
        message += " ".join([feature_names[i]
                             for i in topic.argsort()[:-n_top_words - 1:-1]])
        print(message)

print_top_words(lda, count_vectorizer.get_feature_names_out(), 10)

Topic #0: like got im rock hard high feel nah work roll
Topic #1: na wan gon im girl just youre know ya like
Topic #2: oh come dance u ohoh alright whoa night sweet let
Topic #3: mi yuh di dem fi gyal nuh dey pon like
Topic #4: day away time wa im world youre ive just ill
Topic #5: yeah baby know ill youre oh just wa like mind
Topic #6: ah man dem round like shot head heavy di kick
Topic #7: dont need know time just make cause care im better
Topic #8: feel know home like wa just come youre gone im
Topic #9: love like heart know youre girl just feel im real
Topic #10: let hey away life run music good head stand wont
Topic #11: god wait believe follow youre great life left u right
Topic #12: im got good know just say little ive cause bad
Topic #13: soul blood forever death life dead im free heart alive
Topic #14: ooh la girl oohooh matter ride make high doesnt got
Topic #15: sing lord jesus holy praise come king die power o
Topic #16: new coming shake party morning town water city brand 

In [28]:
def in_top(pred_list,actual, top_x):
    return True if actual in list(pred_list.keys())[:top_x] else False

def logistic_reg_and_score(df: pd.DataFrame):
    # Split the data into features (X) and target (y)
    X = df[topic_columns]
    y = df['parent_genre']

    # Split into training and test sets (80% training, 20% testing)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Create a pipeline: Vectorizer -> Logistic Regression
    # We use 'lbfgs' solver which is good for multiclass problems
    model = LogisticRegression(solver='lbfgs', max_iter=1000,class_weight='balanced')

    # Train the model
    print("Training model...")
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)

    # Evaluate
    print(f"Accuracy: {accuracy_score(y_test, predictions):.2f}")
    print("\nClassification Report:\n")
    print(classification_report(y_test, predictions))

    # Get the probabilities
    probs = model.predict_proba(X_test)

    # Get the genre names from the model
    genres = model.classes_

    # Create a DataFrame where each column is a Genre
    y_prob = pd.DataFrame(probs, columns=genres, index=X_test.index)
    
    # This creates a dictionary of {Genre: Probability} for the top 3
    y_prob['predicted'] = y_prob.apply(
        lambda row: row.nlargest(3).to_dict(), 
        axis=1
    )

    df_pred = pd.DataFrame({
        'actual': y_test
    })

    df_pred = df_pred.join(y_prob['predicted'])
    for x in range(1,4):
        df_pred[f'top_{x}'] = df_pred.apply(lambda row: in_top(row['predicted'],row['actual'],x), axis=1)

    return df_pred

In [29]:
pred = logistic_reg_and_score(df_songs)

Training model...
Accuracy: 0.25

Classification Report:

                  precision    recall  f1-score   support

       Asian Pop       0.37      0.28      0.32        96
       Classical       0.04      0.39      0.08        44
      Electronic       0.48      0.15      0.23       882
    Folk/Country       0.28      0.28      0.28       322
   Hip-Hop & R&B       0.06      0.62      0.11        29
    Jazz & Blues       0.04      0.21      0.07        57
           Metal       0.51      0.57      0.54       452
             Pop       0.15      0.09      0.11       285
Reggae/Caribbean       0.33      0.44      0.38        86
            Rock       0.24      0.05      0.09       493
       Soul/Funk       0.11      0.21      0.15       157
  World/Regional       0.48      0.59      0.53       172

        accuracy                           0.25      3075
       macro avg       0.26      0.32      0.24      3075
    weighted avg       0.35      0.25      0.26      3075

